In [0]:
CREATE OR REPLACE FUNCTION cineai.app.get_screenings(
  p_movie_id STRING COMMENT 'Movie ID or empty string',
  p_city STRING COMMENT 'City or empty string',
  p_date STRING COMMENT 'YYYY-MM-DD or empty string',
  p_cinema STRING COMMENT 'Cinema name or empty string'
)
RETURNS STRING
COMMENT 'Searches screening records. Inputs: optional movie ID, city, date, and cinema filters; use empty strings to omit filters. Output: JSON array of up to ten screenings. Failure or no matches: empty array.'
RETURN
SELECT to_json(
  slice(
    collect_list(named_struct(
      'screening_id', screening_id,
      'movie_id', movie_id,
      'cinema_name', cinema_name,
      'screening_date', screening_date,
      'start_time', start_time,
      'seats_available', seats_available,
      'status', status
    )),
    1,
    10
  )
)
FROM cineai.raw.screenings
WHERE (p_movie_id = '' OR movie_id = p_movie_id)
  AND (p_city = '' OR lower(city) = lower(p_city))
  AND (p_date = '' OR cast(screening_date AS STRING) = p_date)
  AND (p_cinema = '' OR lower(cinema_name) = lower(p_cinema));